# Create Synced Tables for Lakebase OLTP

This notebook syncs Delta tables to Lakebase for low-latency OLTP access from Databricks Apps.

**Inputs:**
- Delta tables (Bronze layer):
  - articles
  - customers
  - transactions

- Delta tables (Gold layer):
  - product_sales_summary
  - customer_demographics
  - time_series_sales

**Outputs:**
- Synced tables (Lakebase OLTP):
  - articles_synced
  - customers_synced
  - transactions_synced
  - product_sales_summary_synced
  - customer_demographics_synced
  - time_series_sales_synced

**Configuration:**
- **Lakebase Instance:** shared-online-store
- **Catalog Type:** Standard Unity Catalog (jongseob_demo)
- **Sync Mode:** SNAPSHOT (one-time sync, no continuous updates)
  - Best for standard catalogs and tables that don't require real-time updates
  - Lower cost compared to TRIGGERED or CONTINUOUS modes
  - To refresh data, manually re-run this notebook

**Note:** The transactions table uses `transaction_id` (synthetic unique key) as the primary key for syncing.


In [ ]:
%pip install -U databricks-sdk
%restart_python

## Setup


In [ ]:
import sys
sys.path.append("..")

import mlflow
from config.paths import MLFLOW_EXPERIMENT_DATA
from config.catalog_config import *
from fashion_rec_dataops.synced_table_utils import create_multiple_synced_tables


In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_DATA)


In [ ]:
# Check all source tables exist
source_tables = [
    # Bronze tables
    f"{CATALOG}.{SCHEMA}.articles",
    f"{CATALOG}.{SCHEMA}.customers",
    f"{CATALOG}.{SCHEMA}.transactions",
    # Gold feature tables
    f"{CATALOG}.{SCHEMA}.product_sales_summary",
    f"{CATALOG}.{SCHEMA}.customer_demographics",
    f"{CATALOG}.{SCHEMA}.time_series_sales"
]

print("Verifying source tables:")
print("=" * 60)
for table in source_tables:
    try:
        count = spark.table(table).count()
        print(f"✓ {table}: {count:,} rows")
    except Exception as e:
        print(f"✗ {table}: ERROR - {str(e)}")
        raise Exception(f"Source table {table} does not exist or is not accessible")
print("=" * 60)


## Configure Synced Tables

Define all synced table configurations with appropriate primary keys.


In [ ]:
# Define synced table configurations
synced_table_configs = [
    # Bronze tables - Raw data synced for app access
    {
        "source_table": f"{CATALOG}.{SCHEMA}.articles",
        "synced_table": f"{CATALOG}.{SCHEMA}.articles_synced",
        "primary_key": ["article_id"]
    },
    {
        "source_table": f"{CATALOG}.{SCHEMA}.customers",
        "synced_table": f"{CATALOG}.{SCHEMA}.customers_synced",
        "primary_key": ["customer_id"]
    },
    {
        "source_table": f"{CATALOG}.{SCHEMA}.transactions",
        "synced_table": f"{CATALOG}.{SCHEMA}.transactions_synced",
        "primary_key": ["transaction_id"]  # Synthetic unique key added during data loading
    },
    # Gold feature tables - Aggregated features for dashboard
    {
        "source_table": f"{CATALOG}.{SCHEMA}.product_sales_summary",
        "synced_table": f"{CATALOG}.{SCHEMA}.product_sales_summary_synced",
        "primary_key": ["article_id"]
    },
    {
        "source_table": f"{CATALOG}.{SCHEMA}.customer_demographics",
        "synced_table": f"{CATALOG}.{SCHEMA}.customer_demographics_synced",
        "primary_key": ["customer_id"]
    },
    {
        "source_table": f"{CATALOG}.{SCHEMA}.time_series_sales",
        "synced_table": f"{CATALOG}.{SCHEMA}.time_series_sales_synced",
        "primary_key": ["date"]
    }
]

print(f"Configured {len(synced_table_configs)} tables for sync to Lakebase")
print(f"Lakebase instance: {LAKEBASE_INSTANCE}")
print()
for config in synced_table_configs:
    print(f"  • {config['source_table']} → {config['synced_table']}")
    print(f"    Primary key: {', '.join(config['primary_key'])}")


## Create All Synced Tables

Use the `create_multiple_synced_tables` utility to batch create all synced tables.

**Note:** If a synced table already exists, it will be deleted and recreated to ensure proper synchronization. The system will wait for deletion to complete before creating the new table.


In [ ]:
print("=" * 60)
print(f"Creating synced tables in Lakebase instance: {LAKEBASE_INSTANCE}")
print("=" * 60)
print()

# Create all synced tables (delete and recreate if exists)
# Using SNAPSHOT mode for standard catalog - one-time sync, no continuous updates
synced_results = create_multiple_synced_tables(
    table_configs=synced_table_configs,
    lakebase_instance=LAKEBASE_INSTANCE,
    skip_if_exists=False,  # Delete and recreate tables if they exist
    scheduling_policy="SNAPSHOT"  # SNAPSHOT mode: one-time sync (best for standard catalogs)
)

print()
print("=" * 60)


## Verify and Summary


In [ ]:
# Check results
successful = sum(1 for success in synced_results.values() if success)
failed = len(synced_table_configs) - successful

print("SYNC RESULTS")
print("=" * 60)
for synced_table, success in synced_results.items():
    status = "✓ SUCCESS" if success else "✗ FAILED"
    print(f"{status}: {synced_table}")
print("=" * 60)
print(f"Total: {successful}/{len(synced_table_configs)} successful")

if failed > 0:
    print(f"\n⚠ {failed} synced table(s) failed to create")
    raise Exception("Some synced tables failed to create")
else:
    print(f"\n✓ All {successful} synced tables created successfully!")


## Log to MLflow


In [ ]:
with mlflow.start_run(run_name="create_synced_tables") as run:
    
    # Log synced table details
    mlflow.log_param("lakebase_instance", LAKEBASE_INSTANCE)
    mlflow.log_param("num_synced_tables", len(synced_table_configs))
    mlflow.log_metric("successful_syncs", successful)
    mlflow.log_metric("failed_syncs", failed)
    
    # Log individual table statuses
    for i, config in enumerate(synced_table_configs, 1):
        source = config['source_table'].split('.')[-1]
        synced = config['synced_table'].split('.')[-1]
        success = synced_results[config['synced_table']]
        mlflow.log_param(f"table_{i}_{source}", f"{synced} ({'success' if success else 'failed'})")
    
    mlflow.set_tag("stage", "synced_tables")
    
    print("\n" + "="*60)
    print("SYNCED TABLES SUMMARY")
    print("="*60)
    print(f"Lakebase Instance: {LAKEBASE_INSTANCE}")
    print(f"Total Synced Tables: {successful}/{len(synced_table_configs)}")
    print("\nBronze Tables Synced:")
    for config in synced_table_configs[:3]:
        print(f"  ✓ {config['synced_table']}")
    print("\nGold Feature Tables Synced:")
    for config in synced_table_configs[3:]:
        print(f"  ✓ {config['synced_table']}")
    print("="*60)
